# RQ3 — Mixed (CAG)10/(CAG)47 systems

In [ ]:
from pathlib import Path
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"


In [ ]:
from src.plots.figures import (
    plot_largest_contact_composition, plot_largest_cluster_metric,
    plot_ranked_condition,
)

MIXTURE_SWEEP = "52b_from_frame"
LENGTH_SWEEP = "51a_repeat_number"
DENSITY = 0.01
STIFFNESS = 2.5
E_VALUES = (2, 4, 8)

mixture = filter_rows(
    load_sweep(OUTPUT_ROOT, MIXTURE_SWEEP, "summary"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES,
)
pure = filter_rows(
    load_sweep(OUTPUT_ROOT, LENGTH_SWEEP, "summary"),
    density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES,
)

## Ranked-cluster composition — pure and mixed systems

In [ ]:
fig, axes = plt.subplots(3, len(E_VALUES), figsize=(15, 10),
                         layout="constrained", sharey=True)
pure_47 = filter_rows(pure, repeat_count=47)
pure_10 = filter_rows(pure, repeat_count=10)

for column, strength in enumerate(E_VALUES):
    plot_ranked_condition(
        filter_rows(pure_47, E_att=strength), axes[0, column],
        style=STYLE, color=STYLE.species_colors["cag47"],
    )
    plot_ranked_condition(
        filter_rows(pure_10, E_att=strength), axes[1, column],
        style=STYLE, color=STYLE.species_colors["cag10"],
    )
    plot_ranked_condition(
        filter_rows(mixture, E_att=strength), axes[2, column],
        style=STYLE, species=("cag10", "cag47"), annotate_molecules=True,
    )
    axes[0, column].set_title(rf"$E_{{\mathrm{{att}}}} = {strength:g}$")
    for row in range(3):
        if row < 2:
            axes[row, column].set_xlabel("")
        if column != 0:
            axes[row, column].set_ylabel("")

for row, label in enumerate(("Pure (CAG)47", "Pure (CAG)10", "Mixture")):
    axes[row, 0].annotate(label, xy=(-.33, .5), xycoords="axes fraction",
                          va="center", ha="center", rotation=90)
handles, labels = axes[2, -1].get_legend_handles_labels()
axes[2, -1].legend(handles, labels, loc="upper left",
                     bbox_to_anchor=(1.02, 1), title="RNA species", frameon=False)
plt.show()

## Contact composition of the largest cluster

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), layout="constrained")
plot_largest_contact_composition(mixture, ax, style=STYLE,
                                 annotate_counts=True, annotate_total=True)
plt.show()

## Largest-cluster size and connectivity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
plot_largest_cluster_metric(
    mixture, axes[0], metric="largest_fraction_total_particles",
    annotation="largest_n_molecules", annotation_label="Molecules",
    style=STYLE, color="#286D9E",
)
plot_largest_cluster_metric(
    mixture, axes[1], metric="largest_mean_degree",
    annotation="largest_n_edges", annotation_label="Edges",
    style=STYLE, color="#BF7849",
)
plt.show()

## Molecular-contact network

In [ ]:
EXPORT_PYVIS = False
if EXPORT_PYVIS:
    from src.plots.networks import save_pyvis_network

    reps = filter_rows(
        load_sweep(OUTPUT_ROOT, MIXTURE_SWEEP, "replicates"),
        density=DENSITY, angle_k_ppp=STIFFNESS, E_att=E_VALUES[0],
    )
    first = reps.iloc[0]
    name = first["folder_name"]
    stem = first["traj_replicate"]
    source_root = ROOT.parent / "validation" / "bulk" / "output_dir"
    output_run = OUTPUT_ROOT / MIXTURE_SWEEP / name
    input_run = source_root / MIXTURE_SWEEP / name

    def existing_archive(relative_path):
        for directory in (output_run, input_run):
            candidate = directory / relative_path
            if candidate.is_file():
                return candidate
        raise FileNotFoundError(relative_path)

    graph = existing_archive(
        f"computed_metrics/mol_graph/{stem}_molecule_contact_edges.npz"
    )
    lookup = existing_archive("computed_metrics/pairing/molecule_lookup.npy")
    html = save_pyvis_network(
        graph, lookup,
        OUTPUT_ROOT / MIXTURE_SWEEP / "figures" / "mixture_network.html",
        style=STYLE, species_by_length={30: "cag10", 141: "cag47"},
    )
    print(html)